<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 11.2: Building & Using an Agent Tool - Answers

A large language model can only *talk* until you give it **tools** — plain
Python functions it can call to actually *do* things. In this lab you'll take a
function that scores how suspicious a domain looks and turn it into a tool that
Claude can call on its own.

The security scoring is given to you. The point of this lab is the **mechanics
of tool use**: writing the docstring the model reads, binding the tool to an
agent, invoking it, and watching the agent decide to call it.


In [12]:
# Load Libraries - Make sure to run this cell!
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

# Loads ANTHROPIC_API_KEY from your .env file in the project root.
load_dotenv()

MODEL = "claude-opus-4-8"


## The function you'll wrap

First, a plain function. Nothing AI about it yet — it takes a domain string and
returns a verdict string. Run it and confirm it works.

In [13]:
# GIVEN — you do NOT need to understand this deeply.
# A few cheap, explainable heuristics that flag algorithmically-generated
# ("DGA") or otherwise sketchy domains. This is the same idea as Worksheet 4.1.
from collections import Counter
import math

def _shannon_entropy(s: str) -> float:
    """Randomness of a string. DGA domains score high (~3.5+)."""
    if not s:
        return 0.0
    return -sum((n / len(s)) * math.log2(n / len(s)) for n in Counter(s).values())

def score_domain(domain: str) -> str:
    """Return a plain-string verdict + reasons for a domain."""
    domain = domain.strip().lower()
    for prefix in ("https://", "http://"):
        if domain.startswith(prefix):
            domain = domain[len(prefix):]
    domain = domain.split("/")[0]

    label = domain.split(".")[0]
    reasons = []
    entropy = _shannon_entropy(label)
    if entropy > 3.5:
        reasons.append(f"high entropy ({entropy:.2f}) — possible DGA")
    if len(domain) > 30:
        reasons.append(f"unusually long ({len(domain)} chars)")
    digits = sum(ch.isdigit() for ch in domain)
    if digits and digits / len(domain) > 0.3:
        reasons.append(f"digit-heavy ({digits} digits)")
    if domain.rsplit(".", 1)[-1] in {"zip", "mov", "xyz", "top", "tk"}:
        reasons.append("high-abuse TLD")

    verdict = "SUSPICIOUS" if reasons else "likely benign"
    detail = "; ".join(reasons) if reasons else "no red flags"
    return f"{domain}: {verdict}. {detail}"

# Sanity check — it's just a function that takes a string and returns a string.
print(score_domain("google.com"))
print(score_domain("kq3v9z7bnx4p2wlm8.top"))


google.com: likely benign. no red flags
kq3v9z7bnx4p2wlm8.top: SUSPICIOUS. high entropy (4.09) — possible DGA; high-abuse TLD


## Step 1 — Turn the function into a tool

The `@tool` decorator (from LangChain) turns a normal function into something an
agent can call. The function's **docstring becomes the tool's description** —
this is the text Claude reads to decide *whether* and *how* to use it. A vague
docstring means the agent won't know when to call your tool.

In [14]:
# Turn score_domain into something Claude can call.
# The DOCSTRING is not a comment — it is the tool's API. The agent reads it to
# decide WHEN to call the tool and WHAT to pass. Write it for the model.
@tool
def check_domain_reputation(domain: str) -> str:
    """Assess whether a domain name looks suspicious or malicious.

    Use this when a user asks whether a domain or URL is safe, or reports a
    suspicious link. Pass a single domain (e.g. "example.com"); returns a
    verdict of SUSPICIOUS or likely benign, with the reasons.
    """
    return score_domain(domain)

# Self-check: @tool wraps the function and exposes its name + description.
assert check_domain_reputation.name == "check_domain_reputation"
assert check_domain_reputation.description  # the docstring the agent reads
print("Tool name:", check_domain_reputation.name)
print("What the agent sees:\n", check_domain_reputation.description)


Tool name: check_domain_reputation
What the agent sees:
 Assess whether a domain name looks suspicious or malicious.

    Use this when a user asks whether a domain or URL is safe, or reports a
    suspicious link. Pass a single domain (e.g. "example.com"); returns a
    verdict of SUSPICIOUS or likely benign, with the reasons.


## Step 2 — Build an agent that has the tool

`create_agent` wires your Claude model together with a list of tools. The
agent will reason about each question and call a tool only when it helps.

In [15]:
# Give the model your tool. create_agent lets Claude decide, on its own,
# whether a question needs the tool.
llm = ChatAnthropic(model=MODEL, api_key=os.getenv("ANTHROPIC_API_KEY"))
agent = create_agent(llm, [check_domain_reputation])
print("Agent ready with tools:", [check_domain_reputation.name])


Agent ready with tools: ['check_domain_reputation']


## Step 3 — Use it

Ask a question. You pass a *user message*, not a function call — the agent looks
at your tool's description and decides for itself to call it.

In [16]:
# Ask the agent about a sketchy-looking domain. Notice you never call the tool
# yourself — the agent reads your docstring and decides to call it.
result = agent.invoke(
    {"messages": [("user", "Is kq3v9z7bnx4p2wlm8.top safe to click?")]}
)
print(result["messages"][-1].content)


**No — I would not recommend clicking that link.** The domain came back as **SUSPICIOUS** for a couple of reasons:

1. **High entropy (randomized-looking name):** The string `kq3v9z7bnx4p2wlm8` is a jumble of random letters and numbers. This is a hallmark of a **DGA (Domain Generation Algorithm)** — a technique malware and phishing operations use to auto-generate throwaway domains that are hard to blocklist. Legitimate businesses almost never use domains like this.

2. **High-abuse TLD:** The `.top` top-level domain is frequently associated with spam, phishing, and malware because it's cheap and lightly regulated.

**My recommendation:**
- **Don't click it**, and don't enter any credentials or personal info if you already have.
- If you received it via email or text, treat the message as likely **phishing** — delete it and don't reply.
- If you already clicked it or entered any information, consider changing any passwords you may have exposed and run a malware scan on your device.
- If

## Step 4 — See the tool call

The final answer hides the interesting part. Walk the message history to see
that the agent actually *chose* to call your tool, what argument it passed
(pulled from the docstring you wrote), and what came back.

In [17]:
# GIVEN — walk the message history so you can SEE that the agent chose to
# call your tool, what it passed in, and what came back.
for msg in result["messages"]:
    role = msg.__class__.__name__
    if role == "AIMessage" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[agent decided to call] {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"[tool returned]           {msg.content}")
    elif role == "AIMessage" and msg.content:
        print(f"[final answer]            {msg.content}")


[agent decided to call] check_domain_reputation({'domain': 'kq3v9z7bnx4p2wlm8.top'})
[tool returned]           kq3v9z7bnx4p2wlm8.top: SUSPICIOUS. high entropy (4.09) — possible DGA; high-abuse TLD
[final answer]            **No — I would not recommend clicking that link.** The domain came back as **SUSPICIOUS** for a couple of reasons:

1. **High entropy (randomized-looking name):** The string `kq3v9z7bnx4p2wlm8` is a jumble of random letters and numbers. This is a hallmark of a **DGA (Domain Generation Algorithm)** — a technique malware and phishing operations use to auto-generate throwaway domains that are hard to blocklist. Legitimate businesses almost never use domains like this.

2. **High-abuse TLD:** The `.top` top-level domain is frequently associated with spam, phishing, and malware because it's cheap and lightly regulated.

**My recommendation:**
- **Don't click it**, and don't enter any credentials or personal info if you already have.
- If you received it via email or text,

## From tool to skill

You just built a **tool**: a single capability the agent can call. A **skill**
is the layer above it — the *analyst judgment and house format* wrapped around
one or more tools.

Look at `../ai_examples/skills/triage-suspicious-url/SKILL.md`. It doesn't add
any new capability. It tells the agent **when** to reach for a reputation
check, to **defang** indicators before echoing them (`evil.com` → `evil[.]com`),
and what **verdict format** to emit. Same tool, wrapped in a repeatable workflow.

- **Tool** = *what the agent can do* (your `check_domain_reputation`).
- **Skill** = *when and how it should do it* (the SOC triage playbook).


In [18]:
# Read the paired skill that would wrap this tool in a real SOC workflow.
from pathlib import Path
print(Path("../ai_examples/skills/triage-suspicious-url/SKILL.md").read_text())


---
name: triage-suspicious-url
description: Use when a link is reported by a user or flagged in an email, alert, or log. Guides safe defensive triage of a suspicious URL or domain and produces a structured verdict with IOCs.
---

# triage-suspicious-url

Defensive SOC workflow for triaging a suspicious URL or domain. The agent
already has the tools (sandbox detonation, whois, TLS/cert lookup, threat
intel). This skill is the analyst judgment and house format wrapped around
them.

## When to use

- A user reports a suspicious link.
- A URL or domain is flagged by email security, a proxy log, or an alert.
- You are asked to decide whether a domain is safe to allow.

## Rules (read first)

- **Never open the URL in a live browser** or any tool that fetches it for
  real. Use the sandbox/detonation tool only.
- **Defang every network indicator** before echoing it anywhere a client
  might auto-link it: `http` becomes `hxxp`, `.` becomes `[.]`
  (so `evil.com` becomes `evil[.]com`).
- Trea